In [3]:
import numpy as np
import scipy.special as special
from sklearn.cluster import DBSCAN


class SACSDBSCAN:
    """
    Self-Adaptive Cuckoo Search using DBSCAN Machine Learning (SACSDBSCAN)
    Paper implementation: A Self-Adaptive Cuckoo Search Algorithm Using a Machine Learning Technique
    """
    def __init__(self, obj_function, dim, lower_bound=0, upper_bound=1,
                 initial_nests=10, max_iter=1000, alpha=0.01, beta=1.5,
                 free_exec_param=100, eps=0.9, min_samples=4):

        self.obj_func = obj_function
        self.dim = dim
        self.lb = lower_bound
        self.ub = upper_bound

        self.n_nests = initial_nests
        self.max_iter = max_iter
        self.alpha = alpha
        self.beta = beta
        self.Pa = 0.1

        self.eps = eps
        self.min_samples = min_samples
        self.free_exec_param = free_exec_param

        self.stagnation_counter = 0
        self.improvement_counter = 0
        self.last_best_fit = float('inf')

    def _transfer_function(self, continuous_val):
        """Sigmoid binarization transfer function for discrete space."""
        sigmoid = 1 / (1 + np.exp(-continuous_val))
        return (np.random.rand(*sigmoid.shape) < sigmoid).astype(int)

    def _levy_flight(self, shape):
        """Generates random step lengths drawn from a Lévy distribution."""

        num = special.gamma(1 + self.beta) * np.sin(np.pi * self.beta / 2)
        den = special.gamma((1 + self.beta) / 2) * self.beta * (2 ** ((self.beta - 1) / 2))
        sigma_u = (num / den) ** (1 / self.beta)
        sigma_v = 1

        u = np.random.normal(0, sigma_u, size=shape)
        v = np.random.normal(0, sigma_v, size=shape)
        step = u / (np.abs(v) ** (1 / self.beta))
        return step

    def optimize(self):
        population = np.random.randint(2, size=(self.n_nests, self.dim))
        fitness = np.array([self.obj_func(nest) for nest in population])

        best_idx = np.argmin(fitness)
        global_best_fit = fitness[best_idx]
        global_best_sol = population[best_idx].copy()
        self.last_best_fit = global_best_fit

        print(f"Initial Best Fitness: {global_best_fit}")

        for t in range(1, self.max_iter + 1):

            for i in range(self.n_nests):
                if np.random.rand() > self.Pa:
                    population[i] = np.random.randint(2, size=self.dim)
                    fitness[i] = self.obj_func(population[i])

            for i in range(self.n_nests):
                if np.random.rand() <= self.Pa:
                    step_size = self.alpha * self._levy_flight(self.dim)
                    cont_sol = population[i] + step_size
                    new_bin_sol = self._transfer_function(cont_sol)
                    new_fit = self.obj_func(new_bin_sol)

                    if new_fit < fitness[i]:
                        population[i] = new_bin_sol
                        fitness[i] = new_fit

            min_fit_idx = np.argmin(fitness)
            min_fit = fitness[min_fit_idx]

            if min_fit < global_best_fit:
                global_best_fit = min_fit
                global_best_sol = population[min_fit_idx].copy()

            if t > 1 and t % self.free_exec_param == 0:
                db = DBSCAN(eps=self.eps, min_samples=self.min_samples, metric='hamming').fit(population)
                labels = db.labels_

                n_noise = np.sum(labels == -1)
                noise_percentage = n_noise / self.n_nests

                if noise_percentage > 0.4:
                    self.Pa = 0.4
                elif noise_percentage < 0.1:
                    self.Pa = 0.1
                else:
                    self.Pa = noise_percentage

                if global_best_fit == self.last_best_fit:
                    self.stagnation_counter += 1
                    self.improvement_counter = 0
                else:
                    self.improvement_counter += 1
                    self.stagnation_counter = 0
                    self.last_best_fit = global_best_fit

                if self.stagnation_counter >= 4:
                    if self.n_nests < 50:
                        new_nests = np.random.randint(2, size=(5, self.dim))
                        new_fit = np.array([self.obj_func(n) for n in new_nests])
                        population = np.vstack((population, new_nests))
                        fitness = np.concatenate((fitness, new_fit))
                        self.n_nests += 5
                    self.stagnation_counter = 0

                elif self.improvement_counter >= 4:
                    if self.n_nests > 10:
                        sorted_indices = np.argsort(fitness)
                        population = population[sorted_indices[:-5]]
                        fitness = fitness[sorted_indices[:-5]]
                        self.n_nests -= 5
                    self.improvement_counter = 0

                unique_labels = set(labels) - {-1}
                for k in unique_labels:
                    cluster_indices = np.where(labels == k)[0]
                    cluster_fits = fitness[cluster_indices]
                    expl_infl = np.sum(cluster_fits) / 11.0

                    if np.abs(expl_infl - global_best_fit) > 1.0:
                        renew_count = max(1, len(cluster_indices) // 2)
                        for idx in cluster_indices[:renew_count]:
                            population[idx] = np.random.randint(2, size=self.dim)
                            fitness[idx] = self.obj_func(population[idx])

                print(f"Iteration {t}/{self.max_iter} | Best Fit: {global_best_fit:.2f} | Nests: {self.n_nests} | Pa: {self.Pa:.2f}")

        return global_best_sol, global_best_fit

if __name__ == "__main__":
    np.random.seed(42)

    n_rows, n_cols = 50, 100
    coverage_matrix = np.random.choice([0, 1], size=(n_rows, n_cols), p=[0.85, 0.15])
    costs = np.random.randint(10, 100, size=n_cols)

    def scp_objective_function(x):
        """Set Covering Problem Objective Function with Penalty."""
        if np.sum(x) == 0:
            return 1e9

        covered_rows = np.dot(coverage_matrix, x)
        uncovered_penalty = np.sum(covered_rows < 1) * 10000

        return np.dot(costs, x) + uncovered_penalty

    optimizer = SACSDBSCAN(
        obj_function=scp_objective_function,
        dim=n_cols,
        initial_nests=10,
        max_iter=1000,
        free_exec_param=100
    )

    print("Starting Optimization Process...")
    best_solution, min_cost = optimizer.optimize()

    print("\n==========================================")
    print("        OPTIMIZATION COMPLETED")
    print("==========================================")
    print(f"Minimum Cost Solution Found: {min_cost}")
    print(f"Total Columns Selected    : {np.sum(best_solution)} / {n_cols}")
    print(f"Selected Column Indices   : {np.where(best_solution == 1)[0].tolist()}")
    print("==========================================")

Starting Optimization Process...
Initial Best Fitness: 2176
Iteration 100/1000 | Best Fit: 1687.00 | Nests: 10 | Pa: 0.10
Iteration 200/1000 | Best Fit: 1382.00 | Nests: 10 | Pa: 0.10
Iteration 300/1000 | Best Fit: 1382.00 | Nests: 10 | Pa: 0.10
Iteration 400/1000 | Best Fit: 1319.00 | Nests: 10 | Pa: 0.10
Iteration 500/1000 | Best Fit: 1319.00 | Nests: 10 | Pa: 0.10
Iteration 600/1000 | Best Fit: 1319.00 | Nests: 10 | Pa: 0.10
Iteration 700/1000 | Best Fit: 1319.00 | Nests: 10 | Pa: 0.10
Iteration 800/1000 | Best Fit: 1319.00 | Nests: 15 | Pa: 0.10
Iteration 900/1000 | Best Fit: 1319.00 | Nests: 15 | Pa: 0.10
Iteration 1000/1000 | Best Fit: 1319.00 | Nests: 15 | Pa: 0.10

        OPTIMIZATION COMPLETED
Minimum Cost Solution Found: 1319
Total Columns Selected    : 37 / 100
Selected Column Indices   : [6, 7, 9, 10, 13, 15, 24, 25, 29, 32, 34, 37, 41, 46, 47, 48, 50, 56, 58, 61, 63, 65, 67, 69, 70, 72, 73, 76, 77, 78, 80, 87, 89, 91, 92, 94, 96]
